# Data Augmentation (DA) for Bayesian Probit Regression (Tanner & Wong, 1987)

## Background & Motivation

Suppose we observe binary labels $y_i\in\{0,1\}$ with predictors $x_i\in\mathbb{R}^p$.
A **probit regression** models
$$
\Pr(y_i=1\mid x_i,\beta)=\Phi(x_i^\top\beta),
$$
where $\Phi$ is the standard normal CDF.

Directly sampling from (or even writing down neatly) the posterior
$$
p(\beta\mid y)\propto \prod_{i=1}^n \Phi(x_i^\top\beta)^{y_i}\,[1-\Phi(x_i^\top\beta)]^{1-y_i}\;p(\beta)
$$
can be awkward.

### Data augmentation trick (latent utilities)

Introduce latent variables $z_i$ such that
$$
z_i = x_i^\top\beta + \varepsilon_i,\qquad \varepsilon_i\sim \mathcal{N}(0,1),\qquad
y_i = \mathbf{1}\{z_i>0\}.
$$

Then:

- The **complete-data** posterior is easier to sample.
- With a Gaussian prior $\beta\sim\mathcal{N}(0,\tau^2 I)$, DA becomes a **Gibbs sampler**:
  1) sample $z\mid \beta,y$ (independent truncated normals)  
  2) sample $\beta\mid z$ (multivariate normal)

Under general conditions, the Markov chain $\{\beta^{(t)}\}$ converges to $p(\beta\mid y)$ and the ergodic mean approximates the posterior mean.

We'll apply DA to the **Wisconsin Diagnostic Breast Cancer (WDBC)** dataset and use the posterior mean for classification.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(123)

# Load data
data = load_breast_cancer(as_frame=True)

# Use a compact feature set for speed (you can switch back to all features if desired)
FEATURES = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
    "mean concavity",
]

X_df = data.data[FEATURES]
y = data.target.to_numpy()  # 0 = malignant, 1 = benign

# Re-code to y in {0,1} with "1 = malignant" for the probit model (optional)
y01 = 1 - y

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y01, test_size=0.25, random_state=0, stratify=y01
)

# Standardize features
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

# Add intercept
X_train_design = np.column_stack([np.ones(len(X_train_std)), X_train_std])
X_test_design = np.column_stack([np.ones(len(X_test_std)), X_test_std])

n, p = X_train_design.shape
print("Using features:", FEATURES)
print("Train n =", n, "p =", p, "| malignant rate =", y_train.mean())

## Tasks

### Part A — Theory / Derivation

Assume the probit model with augmentation
$$
z_i\mid \beta \sim \mathcal{N}(x_i^\top\beta,1),\quad y_i=\mathbf{1}\{z_i>0\},
$$
and prior $\beta\sim \mathcal{N}(0,\tau^2 I_p)$.

1. Derive the conditional distribution:
   - $z_i\mid \beta,y_i$ is a **truncated normal**. Specify its truncation region and parameters.
2. Derive $\beta\mid z$ and show it is multivariate normal. Provide its posterior covariance and mean:
   $$
   V = (X^\top X + \tau^{-2}I)^{-1},\qquad m = V X^\top z.
   $$
3. Explain why this algorithm is a special case of the **Gibbs sampler**.

---

### Part B — Implementation / Real data

1. Implement a Gibbs sampler for DA:
   - Sample each $z_i$ from a truncated normal conditional on $(\beta,y_i)$.
   - Sample $\beta$ from its multivariate normal conditional on $z$.
2. Run the chain for a few thousand iterations; discard burn-in; compute:
   - posterior mean of $\beta$;
   - 95% credible intervals for a few selected coefficients.
3. Use the posterior mean $\hat\beta = \mathbb{E}[\beta\mid y]$ to classify test points via
   $$
   \hat p(x)=\Phi(x^\top \hat\beta).
   $$
   Report accuracy and confusion matrix on the test set.

---

### Part C - Tuning

- Monitor mixing using trace plots and autocorrelations for selected coefficients.
- Try a different prior scale $\tau$ and compare posterior uncertainty and test accuracy.

## References

- Tanner, M. A. & Wong, W. H. (1987). *The Calculation of Posterior Distributions by Data Augmentation*. **JASA**, 82(398), 528–550.
- (Dataset) `sklearn.datasets.load_breast_cancer` — Wisconsin Diagnostic Breast Cancer (WDBC).